# LWM2 — Channel-Only Forecasting
- **Model**: LWM2 (element\_length=16, d\_model=64)
- **Task**: SC-wise next-step channel prediction
- **Input**: past `T` channel frames → `(B, T, F_in=2048)`
- **Target**: next channel frame → `(B, F_out=2048)`
- **in\_proj**: F\_in → 512 → 128 → 16 (GELU)

## 1. Imports

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from deepverse import ParameterManager, Dataset as DVDataset
from lwm_model2 import lwm

## 2. Configuration

In [ ]:
# ── 실험 설정 ────────────────────────────────────────────────────
CUDA_ID      = 1
N_SCENES     = 101          # 사용할 scene 수 (0 ~ N_SCENES-1)
N_SC         = 64           # OFDM subcarrier 수
PAST_LEN     = 16           # 입력 시퀀스 길이
BATCH        = 32
EPOCHS       = 500
LR           = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0
TRAIN_RATIO  = 0.75         # train / (train+val)

LOG_PATH     = f"lwm_model2_scene{N_SCENES}.txt"
CKPT_PATH    = f"best_lwm2_scene{N_SCENES}.pt"

DEVICE = torch.device(f"cuda:{CUDA_ID}" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"Scenes : {N_SCENES}  |  N_SC : {N_SC}  |  past_len : {PAST_LEN}")
print(f"Epochs : {EPOCHS}  |  Batch : {BATCH}  |  LR : {LR}")

## 3. DeepVerse 데이터 로드

In [ ]:
scenarios_name = "DT31"
config_path    = f"scenarios/{scenarios_name}/param/config.m"
param_manager  = ParameterManager(config_path)

param_manager.params["scenes"]                           = list(range(N_SCENES))
param_manager.params["comm"]["OFDM"]["selected_subcarriers"] = list(range(N_SC))
param_manager.params["radar"]["enable"]                  = False
for key in ("lidar", "position"):
    if isinstance(param_manager.params.get(key), dict):
        param_manager.params[key]["enable"] = False

dataset     = DVDataset(param_manager)
comm_frames = [d for row in dataset.comm_dataset.data for d in row]
print(f"Total comm frames: {len(comm_frames)}")

## 4. 전처리 유틸리티

In [ ]:
def get_coeffs(frame, ue_idx=0):
    """comm frame → complex coeffs ndarray (N_rx, N_sc)"""
    ue = frame["ue"]
    ch = ue[ue_idx] if isinstance(ue, (list, tuple)) else ue
    if hasattr(ch, "coeffs"):
        return ch.coeffs
    if isinstance(ch, dict) and "coeffs" in ch:
        return ch["coeffs"]
    raise TypeError(f"Cannot get coeffs: ue={type(ue)}, ch={type(ch)}")


def compute_train_minmax(frames, train_t_list, ue_idx=0, eps=1e-16):
    """Train 구간의 real/imag global min-max 계산"""
    r_min = i_min =  float("inf")
    r_max = i_max = -float("inf")
    for t in train_t_list:
        c = get_coeffs(frames[t], ue_idx)
        r_min = min(r_min, float(c.real.min()))
        r_max = max(r_max, float(c.real.max()))
        i_min = min(i_min, float(c.imag.min()))
        i_max = max(i_max, float(c.imag.max()))
    return (r_min, r_max), (i_min, i_max)


def frame_to_tensor(coeffs_np, r_min, r_max, i_min, i_max, device, eps=1e-16):
    """complex coeffs → min-max scaled float tensor, shape (-1,) = (F,)"""
    c = torch.from_numpy(coeffs_np).to(torch.complex64)
    r = (c.real - r_min) / max(r_max - r_min, eps)
    i = (c.imag - i_min) / max(i_max - i_min, eps)
    return torch.cat([r.reshape(-1), i.reshape(-1)]).to(device, dtype=torch.float32)

## 5. Dataset & DataLoader

In [ ]:
class ChannelDataset(Dataset):
    """
    sample[i] = (channel_past, channel_next)
      channel_past : (PAST_LEN, F)  — frames [t-T+1 .. t]
      channel_next : (F,)           — frame  [t+1]
    """
    def __init__(self, frames, time_indices, past_len, device,
                 r_min=0., r_max=1., i_min=0., i_max=1., ue_idx=0):
        self.frames   = frames
        self.past_len = past_len
        self.device   = device
        self.ue_idx   = ue_idx
        self.scale    = (r_min, r_max, i_min, i_max)
        # t >= past_len AND t+1 < N 를 만족하는 인덱스만 허용
        N = len(frames)
        self.indices  = [t for t in time_indices if t >= past_len and t + 1 < N]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t   = self.indices[idx]
        sc  = self.scale
        ch_list = [
            frame_to_tensor(get_coeffs(self.frames[k], self.ue_idx), *sc, self.device)
            for k in range(t - self.past_len + 1, t + 1)
        ]
        ch_past = torch.stack(ch_list, dim=0)                            # (T, F)
        target  = frame_to_tensor(get_coeffs(self.frames[t + 1], self.ue_idx),
                                  *sc, self.device)                      # (F,)
        return ch_past, target

In [ ]:
# ── Train / Val 인덱스 분리 (frame 겹침 없음) ────────────────────
N       = len(comm_frames)
n_train = int(N * TRAIN_RATIO)
train_t = list(range(n_train))
val_t   = list(range(n_train, N))

# Train 구간으로만 min-max 계산
(r_min, r_max), (i_min, i_max) = compute_train_minmax(comm_frames, train_t)
print(f"real : [{r_min:.6e}, {r_max:.6e}]")
print(f"imag : [{i_min:.6e}, {i_max:.6e}]")

scale = (r_min, r_max, i_min, i_max)

train_ds = ChannelDataset(comm_frames, train_t, PAST_LEN, DEVICE, *scale)
val_ds   = ChannelDataset(comm_frames, val_t,   PAST_LEN, DEVICE, *scale)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)

ch, y = next(iter(train_loader))
F_in  = ch.shape[-1]
F_out = y.shape[-1]
print(f"\nTrain: {len(train_ds)}  |  Val: {len(val_ds)}")
print(f"Batch  ch: {tuple(ch.shape)}  y: {tuple(y.shape)}")
print(f"F_in={F_in}, F_out={F_out}")

## 6. Model

In [ ]:
class LWMForecaster(nn.Module):
    """
    LWM backbone (embedding + Transformer layers) 기반 채널 예측 모델
    in_proj : F_in → 512 → 128 → element_length (GELU)
    head    : d_model → F_out
    """
    def __init__(self, base_lwm, F_in, F_out,
                 pool="last", freeze_backbone=False):
        super().__init__()
        self.base = base_lwm
        self.pool = pool

        n_dim   = base_lwm.embedding.element_length  # 16
        d_model = base_lwm.embedding.d_model         # 64

        self.in_proj = nn.Sequential(
            nn.Linear(F_in, 512), nn.GELU(),
            nn.Linear(512, 128),  nn.GELU(),
            nn.Linear(128, n_dim),
        )
        self.head = nn.Linear(d_model, F_out)

        if freeze_backbone:
            for p in self.base.parameters():
                p.requires_grad = False

    def forward(self, ch):
        x = self.in_proj(ch)               # (B, T, n_dim)
        x = self.base.embedding(x)         # (B, T, d_model)
        for layer in self.base.layers:
            x, _ = layer(x)
        z = x[:, -1, :] if self.pool == "last" else x.mean(dim=1)
        return self.head(z)                # (B, F_out)

In [ ]:
backbone = lwm().to(DEVICE)
model    = LWMForecaster(backbone, F_in=F_in, F_out=F_out,
                         pool="last", freeze_backbone=False).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params:,}")

# sanity check
with torch.no_grad():
    yhat = model(ch.to(DEVICE))
print(f"yhat: {tuple(yhat.shape)}  (expected ({BATCH}, {F_out}))")

## 7. 학습 유틸리티

In [ ]:
def nmse_db(yhat, y, eps=1e-16):
    num = torch.sum((yhat - y) ** 2, dim=1)
    den = torch.sum(y ** 2, dim=1).clamp_min(eps)
    return (10.0 * torch.log10((num / den).clamp_min(eps))).mean().item()


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = total_nmse = n = 0
    for ch, y in loader:
        yhat = model(ch)
        loss = F.mse_loss(yhat, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item()
        total_nmse += nmse_db(yhat.detach(), y)
        n += 1
    return total_loss / max(n, 1), total_nmse / max(n, 1)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = total_nmse = n = 0
    for ch, y in loader:
        yhat = model(ch)
        loss = F.mse_loss(yhat, y)
        total_loss += loss.item()
        total_nmse += nmse_db(yhat, y)
        n += 1
    return total_loss / max(n, 1), total_nmse / max(n, 1)

## 8. Optimizer / Scheduler

In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 9. 학습

In [ ]:
best_val   = float("inf")
best_nmse  = None
best_epoch = None
t0_total   = time.time()

with open(LOG_PATH, "w", encoding="utf-8") as f:
    header = (f"=== LWM2 | Channel-Only | scenes={N_SCENES}"
              f" | train={len(train_ds)} | val={len(val_ds)} ===")
    f.write(header + "\n")
    print(header)

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_nmse = train_one_epoch(model, train_loader, optimizer)
        va_loss, va_nmse = evaluate(model, val_loader)
        scheduler.step()
        elapsed = time.time() - t0

        line = (f"[{epoch:03d}/{EPOCHS}] "
                f"train loss={tr_loss:.6f}, nmse(dB)={tr_nmse:.4f} | "
                f"val loss={va_loss:.6f}, nmse(dB)={va_nmse:.4f} | "
                f"{elapsed:.1f}s")
        print(line)
        f.write(line + "\n")

        if va_loss < best_val:
            best_val   = va_loss
            best_nmse  = va_nmse
            best_epoch = epoch
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "F_in": F_in, "F_out": F_out,
                "best_val_loss": best_val, "best_val_nmse_db": best_nmse,
            }, CKPT_PATH)
            ck = f"  -> saved {CKPT_PATH} | best@{epoch}: val nmse={best_nmse:.4f}dB"
            print(ck)
            f.write(ck + "\n")

        f.flush()

    total = time.time() - t0_total
    h, m, s = int(total//3600), int((total%3600)//60), total%60
    summary = (f"\n=== Done === {h}h {m}m {s:.0f}s"
               f" | Best epoch {best_epoch}: val loss={best_val:.6f}"
               f" | val nmse={best_nmse:.4f}dB")
    print(summary)
    f.write(summary + "\n")

print(f"\nLog  → {os.path.abspath(LOG_PATH)}")
print(f"Ckpt → {os.path.abspath(CKPT_PATH)}")